# Frontend Pipeline For Qwen3 MoE

实现 Qwen3-MoE decoder layer 的最小前端全流程，并在 `H100_SXM` 上完成 macro op 仿真。

## 流程
- 加载 `qwen3-moe-235B` model card 并初始化原始配置对象与 `Qwen3MoEModelConfig`
- 初始化 `NandConfig`、`InferenceConfig`、`MoEParallelConfig`
- 初始化 `Qwen3MoEDecoderLayer`
- 使用 `NxTracer` trace 得到计算图
- 运行 `NormalizePass` 获得简化图
- 用 `build_kv_cache_state` 构造 `KVCacheState`
- 构造 `NxGraphMeta` 并写入 `graph.meta`
- 运行 `CodeGenPass` 生成 `macro_op_list`
- 在 `H100_SXM` 上运行完整硬件仿真并校验最终 cycle

In [ ]:
import json
import logging
from collections import Counter
from pathlib import Path

import torch
from torch.fx import GraphModule

from nandmachine.commands.macro import All2AllOp, FlashAttnOp, MatMulOp, VectorOp
from nandmachine.config.config import NandConfig
from nandmachine.config.inference_config import InferenceConfig, MoEParallelConfig
from nandmachine.config.model_config import Qwen3MoEModelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.qwen3_moe import Qwen3MoEDecoderLayer
from nandmachine.frontend.utlis import build_kv_cache_state
from nandmachine.simulator.entry_point import run_macro_ops

MODEL_CARD_PATH = Path("model_cards/qwen3-moe-235B.json")
DEVICE_NAME = "H100_SXM"
COMPILE_MODE = "heuristic-GPU"

assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH


def node_summary(graph_module: GraphModule) -> list[tuple[str, str]]:
    return [(node.op, str(node.target)) for node in graph_module.graph.nodes]


def macro_summary(macro_op_list: list[object]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

In [ ]:
model_card = json.loads(MODEL_CARD_PATH.read_text())
model_card.setdefault("attention_type", "gqa")
raw_model_config = type("Qwen3MoeConfig", (), model_card)()
model_config = Qwen3MoEModelConfig.from_config(raw_model_config)
nand_config = NandConfig(
    num_channels=1,
    num_plane=1,
    num_block=4,
    num_pages=16,
    tRead=1.0,
    tWrite=2.0,
    tErase=3.0,
    page_size=4,
    sram_threshold=1024 * 1024 * 20,
)
parallel_config = MoEParallelConfig(
    num_ranks=8,
    attn_dp_size=8,
    attn_tp_size=1,
    ffn_tp_size=4,
    ffn_ep_size=2,
)
inference_config = InferenceConfig(
    batch_size=2,
    input_sequence_length=8,
    output_sequence_length=4,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes=1024,
    memory_backend="nand",
    parallel_config=parallel_config,
)
kv_cache_state = build_kv_cache_state(nand_config, model_config, inference_config)
graph_meta = NxGraphMeta(
    nand_config=nand_config,
    model_config=model_config,
    inference_config=inference_config,
    kv_cache_state=kv_cache_state,
)
simulator_config = {
    "device_name": DEVICE_NAME,
    "compile_mode": COMPILE_MODE,
}

print(model_config)
print(parallel_config)
print(inference_config)
print(kv_cache_state)
print(simulator_config)

In [ ]:
with torch.device("meta"):
    model = Qwen3MoEDecoderLayer(raw_model_config, parallel_config)
    graph = NxTracer().trace(model)
    graph_module = GraphModule(model, graph)

NormalizePass().transform(graph_module)
graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
CodeGenPass().transform(graph_module)

macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
vector_ops = [op for op in macro_op_list if isinstance(op, VectorOp)]
all_to_all_ops = [op for op in macro_op_list if isinstance(op, All2AllOp)]

print("normalized node count:", len(node_summary(graph_module)))
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("all_to_all count:", len(all_to_all_ops))
print("first 10 vector ops:", [(op.vector_op_type, op.vector_shape) for op in vector_ops[:10]])


In [ ]:
assert macro_op_list
assert any(isinstance(op, MatMulOp) for op in macro_op_list)
assert any(isinstance(op, FlashAttnOp) for op in macro_op_list)
assert any(isinstance(op, All2AllOp) for op in macro_op_list)
assert any(op.vector_op_type == "moe_topk_router" for op in vector_ops)
assert any(op.vector_op_type == "moe_weighted_sum" for op in vector_ops)
assert any(op.vector_op_type == "silu_mul" for op in vector_ops)
assert any(op.vector_op_type == "rms_norm" for op in vector_ops)

print("qwen3 moe frontend pipeline completed successfully")

In [ ]:
macro_op_list[:10]

In [ ]:
len(macro_op_list)

## 完整硬件仿真

直接复用前面生成好的 `nand_config`、`parallel_config`、`inference_config`、`kv_cache_state` 和 `macro_op_list`，在 `H100_SXM` 上运行完整 macro op 仿真。

In [ ]:
final_cycle = run_macro_ops(
    nand_config,
    macro_op_list,
    **simulator_config,
)

assert final_cycle > 0
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("all_to_all count:", len(all_to_all_ops))
print("nand config:", nand_config)
print("parallel config:", parallel_config)
print("inference config:", inference_config)
print("kv cache state:", kv_cache_state)
print("simulator config:", simulator_config)
print("final sim cycle:", final_cycle)